# 01 — Exploratory analysis of the example mutation dataset

This notebook is a quick tour of the bundled example dataset and the
ProteinLM-Bench pipeline. It is intentionally short — the goal is to give
users an interactive entry point before they swap in a real dataset such as
[ProteinGym](https://proteingym.org/) or in-house deep mutational scanning data.

Run from the repository root:

```bash
jupyter notebook notebooks/01_exploratory_analysis.ipynb
```

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

from proteinlm_bench.data import load_mutation_csv, summarize_dataset, train_test_split_variants
from proteinlm_bench.embeddings import embedder_from_dict
from proteinlm_bench.models import RidgeRegressor, RandomForestRegressorWrapper
from proteinlm_bench.metrics import compute_metrics_with_ci
from proteinlm_bench.epistasis import analyze_epistasis, epistasis_summary

## 1. Load and inspect the dataset

In [ ]:
df = load_mutation_csv(ROOT / 'data' / 'example_mutations.csv')
print(df.shape)
df.head()

In [ ]:
summarize_dataset(df)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for n_mut, sub in df.groupby('n_mutations'):
    label = {0: 'WT', 1: 'single', 2: 'double'}.get(n_mut, f'{n_mut}-mut')
    ax.hist(sub['fitness'], bins=15, alpha=0.6, label=label, edgecolor='white')
ax.set_xlabel('Fitness'); ax.set_ylabel('Count'); ax.legend(title='Variant class')
ax.set_title('Fitness distribution by variant class')
plt.show()

## 2. Generate mock embeddings and train a quick regressor

We use the mock backend here so the notebook runs in seconds without downloading PLM weights. Set `backend='huggingface'` to switch to ESM2 or ProtBert.

In [ ]:
train_df, test_df = train_test_split_variants(df, test_size=0.25, seed=0)
embedder = embedder_from_dict({'backend': 'mock', 'embedding_dim': 128})
X_train = embedder.embed_sequences(train_df['mutant_sequence'].tolist())
X_test  = embedder.embed_sequences(test_df['mutant_sequence'].tolist())
y_train = train_df['fitness'].to_numpy(); y_test = test_df['fitness'].to_numpy()
print('X_train:', X_train.shape, ' X_test:', X_test.shape)

In [ ]:
model = RidgeRegressor(alpha=1.0).fit(X_train, y_train)
preds = model.predict(X_test)
results = compute_metrics_with_ci(y_test, preds, bootstrap_samples=200, seed=0)
for name, r in results.items():
    print(f'{name:>20s}  {r.value:+.3f}  CI=[{r.ci_low:+.3f}, {r.ci_high:+.3f}]')

## 3. Epistasis analysis on multi-mutants

We compare predicted fitness for multi-mutants against the additive (single-mutant-sum) expectation. Variants where observed - additive is large in magnitude carry genuine epistatic signal — those are the variants where a sequence-function model that simply learns single-mutant effects will fail.

In [ ]:
epi = analyze_epistasis(train_df, test_df, preds, reference_df=df)
epi

In [ ]:
print(epistasis_summary(epi))

## Next steps

* Swap the synthetic CSV for a real dataset (ProteinGym, MaveDB, or your own DMS screen).
* Switch to a HuggingFace backend (`backend: huggingface`, `model_name: facebook/esm2_t6_8M_UR50D`).
* Run the full benchmark via `python scripts/run_benchmark.py --config configs/default.yaml`.